# RL Env Viz Trials

- https://github.com/newton-physics/newton


## Base Imports


In [2]:
"""Base imports"""
import warnings 
warnings.filterwarnings('ignore')

import argparse, os, time, random
import pickle
from copy import deepcopy
from tqdm import tqdm, trange 

import numpy as np
import matplotlib.pyplot as plt

import torch
from torchvision.transforms import transforms
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, TensorDataset

# ***** Helpers *****
from utils import (
    TIMESTAMP as timestamp, 
    DEVICE, 
    create_folder,
    get_lowest_gpu, 
    Logger
)
create = False # bool to create folder or not

# TODO: put all helper functions below, such as compare model params, etc. 

def printdict(d):
    import json
    print(json.dumps(d, indent=2))

def rprint(x, prefix='', print_content=False):
    """Helper to recursively print dict"""
    if isinstance(x, dict):
        for k, v in x.items():
            print(prefix + k)
            rprint(v, prefix+'   ')
    elif type(x) in [np.ndarray, torch.Tensor, list]:
        if print_content:
            print(f'{prefix}{x}')
        else:
            print(f'{prefix}{type(x)} : {x.shape if type(x) is not list else len(x)}')
    else:
        print(f'{prefix}{x}')

def save_model(model, config, fn):
    torch.save(
        {'model_state_dict': model.state_dict(), 'config': config}, 
        fn
    )

def assert_cuda(module):
    for name, param in module.named_parameters():
        assert param.device == device, f"Param [{name}] is in device [{param.device}] not in [{device}]"
    print(f'Asserting done, all layers are set to device: [{device}]')


def convert_to_tensor(x, store_gpu=True):
    if store_gpu:
        return torch.tensor(np.asarray(x)).float().to(device)
    else:
        return torch.tensor(np.asarray(x)).float()


def set_seed(seed):
    if seed == -1:
        seed = 0 
    print('Setting SEED to', seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        # torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

def printdict(d):
    import json
    print(json.dumps(d, indent=2))

def rprint(x, prefix='', print_content=False):
    """Helper to recursively print dict"""
    if isinstance(x, dict):
        for k, v in x.items():
            print(prefix + k)
            rprint(v, prefix+'   ')
    elif type(x) in [np.ndarray, torch.Tensor, list]:
        if print_content:
            print(f'{prefix}{x}')
        else:
            print(f'{prefix}{type(x)} : {x.shape if type(x) is not list else len(x)}')
    else:
        print(f'{prefix}{x}')

def save_model(model, config, fn):
    torch.save(
        {'model_state_dict': model.state_dict(), 'config': config}, 
        fn
    )

def assert_cuda(module):
    for name, param in module.named_parameters():
        assert param.device == device, f"Param [{name}] is in device [{param.device}] not in [{device}]"
    print(f'Asserting done, all layers are set to device: [{device}]')


def convert_to_tensor(x, store_gpu=True):
    if store_gpu:
        return torch.tensor(np.asarray(x)).float().to(device)
    else:
        return torch.tensor(np.asarray(x)).float()


# set cuda device
DEVICE = get_lowest_gpu(machine='leibniz')
# with open('.cuda.device', 'w') as f: f.write(f'{DEVICE}')
# os.environ['CUDA_VISIBLE_DEVICES'] = str(DEVICE)
device = torch.device(f'cuda:{DEVICE}' if torch.cuda.is_available() else 'cpu')


%load_ext autoreload
%autoreload 2

Using device: 3, with current memory 407 MB


In [2]:
# load cfg 
import yaml
from argparse import Namespace

with open('a_cfg.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# cfg = yaml.safe_load(cfg)
# print(json.dumps(cfg, indent=3))


Namespace(seed=1, env='nonstationary_bandit', envs=100000, envs_eval=100, hists=1, samples=1, horizon=1000, dim=5, lin_d=2, var=0.3, cov=0.0, env_id_start=-1, env_id_end=-1, cp=0.01, ns_type='incremental', expert=False, expert_name='swucb', expert_portion=0.5, normal_portion=1.0, embd=32, head=4, layer=4, lr=0.0005, dropout=0, shuffle=True, train_epochs=200, print_every=5, temporal=True, extreme=False, optimal=False)


## a


Trying to create symlink for tensorboard runs to `records` dir.
Symlinking done.
